In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import bitsandbytes as bnb

model_id = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [27]:
def calculate_score(criteria):
    weights = {
        1: 10,
        2: 15,
        3: 20,
        4: 20,
        5: 15,
        6: 15,
        7: 5
    }

    max_scores = {
        4: 20,
        6: 10
    }

    allowed_scores = {
        1: {"Да", "Нет"},
        2: {"Да", "Нет", "N/A"},
        3: {"Да", "Нет", "N/A"},
        4: {0, 5, 10, 15, 20, "N/A"},
        5: {"Да", "Нет"},
        6: {0, 4, 7, 10},
        7: {"Да", "Нет", "N/A"},
        8: {"Да", "Нет"}
    }

    if len(criteria) != 8:
        raise ValueError("Expected exactly 8 criteria")

    ids = [item["id"] for item in criteria]

    if set(ids) != set(range(1, 9)):
        raise ValueError("Missing or duplicate criterion IDs")

    earned = 0
    applicable_weight = 0
    severe_violation = False

    for item in criteria:
        criterion_id = item["id"]
        score = item["score"]

        if score not in allowed_scores[criterion_id]:
            raise ValueError(
                f"Invalid score {score} for criterion {criterion_id}"
            )

        if criterion_id == 8:
            severe_violation = score == "Да"
            continue

        if score == "N/A":
            continue

        weight = weights[criterion_id]
        applicable_weight += weight

        if criterion_id in max_scores:
            normalized_score = score / max_scores[criterion_id]
        else:
            normalized_score = 1 if score == "Да" else 0

        earned += weight * normalized_score

    total_score = round(
        earned / applicable_weight * 100, 2
    ) if applicable_weight > 0 else None

    return {
        "total_score": total_score,
        "severe_violation": severe_violation
    }

In [29]:
import json
import torch

def evaluate_dialogue(row):
    conversation_id = row["conversation_id"]
    language = row["language"]
    dialogue = row["dialogue_text"]

    user_prompt = f"""
Оцени следующий диалог по чек-листу.

ЧЕК-ЛИСТ:
{checklist}

CONVERSATION ID: {conversation_id}
LANGUAGE: {language}

ДИАЛОГ:
{dialogue}

Верни только валидный JSON следующего формата:

{{
    "conversation_id": "{conversation_id}",
    "criteria": [
        {{
            "id": 1,
            "score": "Да",
            "explanation": "Объяснение оценки",
            "evidence_ids": [9]
        }}
    ]
}}

ВАЖНО:
- Верни все 8 критериев с id от 1 до 8.
- Используй допустимые оценки из чек-листа.
- evidence_ids должны ссылаться только на существующие сообщения.
- Не придумывай факты, которых нет в диалоге.
- Не добавляй Markdown или текст вне JSON.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1500,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[-1]

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    result = json.loads(response)

    if result["conversation_id"] != conversation_id:
        raise ValueError("Conversation ID mismatch")

    scores = calculate_score(result["criteria"])

    result.update(scores)
    result["language"] = language
    result["channel"] = row["channel"]

    return result

In [4]:
import pandas as pd

df = pd.read_json("qwen_dataset.json")

print(df.shape)
df.head()

(65, 4)


,conversation_id,channel,language,dialogue_text
0,p1-ru-001,agent chat,RU,[0] [00:00] client: Где посмотреть мой тариф? ...
1,p1-ru-002,agent chat,RU,[0] [00:00] client: Почему у меня поменялся та...
2,p1-ru-003,agent chat,RU,[0] [00:00] client: Какая комиссия на тарифе и...
3,p1-ru-004,agent chat,RU,[0] [00:00] client: Как пополнить брокерский с...
4,p1-ru-005,agent chat,RU,[0] [00:00] client: Хочу пополнить счёт и не п...


In [23]:
system_prompt = """
Ты — аудитор качества клиентской поддержки Freedom Broker.

Твоя задача — объективно оценивать работу оператора по официальному чек-листу.

ОСНОВНЫЕ ПРАВИЛА:

1. Оценивай исключительно действия оператора (speaker: operator).
   Сообщения client, bot и system используй только как контекст.

2. Не приписывай оператору действия бота или системы.
   Однако системные сообщения могут подтверждать идентификацию,
   завершение или прерывание чата.

3. Используй только информацию из предоставленного диалога.
   Не придумывай факты, реплики, действия или подтверждения.

4. Для каждого критерия сначала проверь применимость.
   N/A разрешено ТОЛЬКО для критериев 2, 3, 4 и 7.

5. N/A означает, что критерий неприменим.
   "Нет" означает, что действие требовалось, но оператор
   не выполнил его или выполнил неправильно.

6. Если для ответа достаточно информации, уже имеющейся
   в истории диалога, оператор НЕ обязан задавать
   дополнительные вопросы.

7. Если клиент уже сообщил нужные сведения боту,
   оператор должен учитывать эти сведения.

8. Не штрафуй оператора за отсутствие действий,
   которые не требовались в конкретной ситуации.

9. Не оценивай порог первой реакции в 3 минуты,
   теги и вложения.

10. Для каждого критерия укажи конкретные evidence_ids.
    Используй только существующие номера реплик.
    Для пропущенного действия укажи реплику, после которой
    это действие ожидалось.
    Для N/A укажи реплики, подтверждающие неприменимость.

11. Если прямого доказательства нет, используй пустой список [].
    Не выдумывай доказательства.

12. Объяснение должно соответствовать выставленной оценке.
    Не добавляй замечания, не влияющие на критерий.

13. Не путай критерии:
    - Консультация оценивает правильность и полноту решения.
    - Управление диалогом оценивает ясность и последовательность.
    - Эмпатия оценивает вежливость и реакцию на проблему.

14. Не вычисляй итоговый балл.
    Итог рассчитывается отдельно в Python.

15. Верни исключительно корректный JSON.
    Без Markdown, пояснений до или после JSON.

16. Не добавляй в explanation гипотетические проблемы,
    которые не произошли в диалоге.

17. Каждое утверждение о действиях оператора должно
    подтверждаться конкретной репликой.
    Не утверждай, что оператор предложил помощь,
    извинился или проверил данные, если этого нет в тексте.

18. При определении N/A для идентификации анализируй
    характер запроса клиента, а не факт наличия
    или отсутствия процедуры идентификации.

19. Если критерий получает максимальную оценку,
    объяснение должно обосновывать именно эту оценку.
    Не добавляй выдуманные недостатки ради критики.

20. В критерии 6 предупреждение требуется только
    при фактическом ожидании ответа.
    Не требуй предупреждений о гипотетических
    проблемах клиента.

21. Не добавляй замечания после слов "но", "однако",
    "хотя", если эти замечания не подтверждены
    конкретными событиями диалога.

22. Для каждого критерия пиши explanation
    длиной не более 20 слов.
    Не повторяй текст критерия.
    Объясняй только причину оценки.
"""

In [12]:
checklist = """
КРИТЕРИЙ 1 — УСТАНОВЛЕНИЕ КОНТАКТА
Вес: 10
Возможные оценки: "Да", "Нет"

"Да":
Оператор поздоровался на языке клиента и представился
в своём первом ответе.

"Нет":
Хотя бы одно из этих условий не выполнено.

N/A запрещено.


КРИТЕРИЙ 2 — ВЫЯВЛЕНИЕ ПОТРЕБНОСТИ
Вес: 15
Возможные оценки: "Да", "Нет", "N/A"

Проверяй в следующем порядке:

Шаг 1:
Было ли достаточно информации для ответа
на момент подключения оператора?

Если ДА:
Поставь "N/A".
Уточняющие вопросы не требовались.

Если НЕТ:
Перейди к шагу 2.

Шаг 2:
Учёл ли оператор уже известные детали и уточнил ли
недостающую информацию до ответа?

Если ДА:
Поставь "Да".

Если НЕТ:
Поставь "Нет".

ВАЖНО:
Не путай учёт уже известных деталей с выявлением потребности.
Если оператор правильно ответил без дополнительных вопросов,
потому что всё уже было известно, ставь "N/A", а не "Да".


КРИТЕРИЙ 3 — ИДЕНТИФИКАЦИЯ
Вес: 20
Возможные оценки: "Да", "Нет", "N/A"

Сначала определи, требовал ли запрос идентификации.

"N/A":
Клиент запрашивает общую информацию или инструкции,
не требующие доступа к защищённым данным.

"Да":
Идентификация требовалась и была подтверждена
до раскрытия персональных данных или выполнения
защищённого действия.

"Нет":
Идентификация требовалась перед раскрытием данных
или защищённым действием, но не была подтверждена.

ВАЖНО:
- Не считай отсутствие идентификации нарушением,
  если она не требовалась.
- Не считай простое предложение пройти идентификацию
  подтверждением её успешного прохождения.
- Учитывай системные сообщения о подтверждении личности.
- Если проверка личности требовалась, но не подтверждена
  в чате, ставь "Нет".


КРИТЕРИЙ 4 — КОРРЕКТНАЯ И ПОЛНАЯ КОНСУЛЬТАЦИЯ
Вес: 20
Возможные оценки: 0, 5, 10, 15, 20, "N/A"

20:
Все существенные вопросы клиента закрыты
или дан корректный следующий шаг.

15:
Основной вопрос решён, но пропущена второстепенная деталь.

10:
Часть запроса закрыта, но существенный вопрос остался открытым.

5:
Есть общая подсказка, но клиенту всё ещё непонятно,
как действовать.

0:
Ответа по существу нет или ключевая рекомендация ошибочна.

"N/A":
Чат закончился до того, как клиент сформулировал запрос.

ВАЖНО:
- Учитывай ВСЕ вопросы клиента, включая дополнительные.
- Не считай вежливость доказательством правильной консультации.
- Если оператор дал корректный следующий шаг,
  необязательно, чтобы проблема уже была окончательно решена.
- Не используй промежуточные оценки, например 12 или 18.


КРИТЕРИЙ 5 — ЭМПАТИЯ И КЛИЕНТООРИЕНТИРОВАННОСТЬ
Вес: 15
Возможные оценки: "Да", "Нет"

"Да":
Оператор вежлив и, если клиент выразил недовольство
или описал неудобство, отреагировал на конкретную проблему.

"Нет":
Оператор невежлив или не отреагировал на выраженное
недовольство либо описанное неудобство.

ВАЖНО:
- Не требуй извинений или сочувственных фраз,
  если ситуация этого не предполагает.
- Не требуй эмоциональных фраз в каждом ответе.
- Оценивай поведение оператора в контексте проблемы клиента.

N/A запрещено.


КРИТЕРИЙ 6 — УПРАВЛЕНИЕ ДИАЛОГОМ,
СКОРОСТЬ И ГРАМОТНОСТЬ
Вес: 15
Возможные оценки: 0, 4, 7, 10

10:
Ответы ясные и связные.
Если было явное ожидание, оператор предупредил.

7:
Один заметный сбой, но клиент может понять ответ
и продолжить.

4:
Клиенту приходится переспрашивать из-за формулировок
или порядка ответов.

0:
Из-за бессвязных или противоречивых реплик
ход диалога непонятен.

ВАЖНО:
- Не штрафуй за отсутствие предупреждения об ожидании,
  если явного ожидания не было.
- Не оценивай порог первой реакции в 3 минуты.
- Не путай неполный ответ с непонятным ответом.
- Используй только значения 0, 4, 7, 10.

N/A запрещено.


КРИТЕРИЙ 7 — ЗАВЕРШЕНИЕ
Вес: 5
Возможные оценки: "Да", "Нет", "N/A"

"Да":
Оператор подвёл итог или предложил дальнейшую помощь
и попрощался.

"Нет":
Оператор имел возможность корректно завершить разговор,
но не выполнил требования.

"N/A":
Чат оборвался до возможности его завершить.

ВАЖНО:
- Если клиент внезапно отключился до завершения,
  используй "N/A".
- Не штрафуй оператора за прерывание связи,
  которое от него не зависело.


КРИТЕРИЙ 8 — ГРУБЫЕ ОШИБКИ
Вес: 0
Возможные оценки: "Да", "Нет"

"Да":
Выявлено хотя бы одно из следующих нарушений:

1. Оскорбление клиента.
2. Предложение платной услуги в обход компании.
3. Раскрытие конфиденциальной информации.

"Нет":
Ни одного из перечисленных нарушений не обнаружено.

ВАЖНО:
- Для предложения платной услуги в обход компании
  достаточно самого предложения.
- Не требуется согласие клиента или перевод денег.
- Не отмечай грубую ошибку только из-за невежливого тона,
  если оскорбления или другого указанного нарушения нет.
- При "Да" обязательно укажи конкретную реплику оператора.

N/A запрещено.
"""

In [21]:
row = df[df["conversation_id"] == "p1-ru-001"].iloc[0]

dialogue = row["dialogue_text"]
conversation_id = row["conversation_id"]
language = row["language"]

user_prompt = f"""
Оцени следующий диалог по официальному чек-листу.

ЧЕК-ЛИСТ:
{checklist}

ИНФОРМАЦИЯ О ДИАЛОГЕ:
conversation_id: {conversation_id}
language: {language}

ДИАЛОГ:
{dialogue}

ИНСТРУКЦИЯ:

Для каждого из 8 критериев:
1. Определи применимость.
2. Выбери оценку строго из разрешённых значений.
3. Объясни решение в 1–2 предложениях.
4. Укажи номера реплик, подтверждающих решение.

Перед формированием ответа проверь:
- N/A используется только там, где разрешено.
- Критерий 2 не получает "Да" только за правильный ответ
  без необходимости уточнений.
- Критерий 3 учитывает системное подтверждение личности.
- Критерии 4 и 6 используют только разрешённые числа.
- Критерий 8 означает наличие грубой ошибки,
  а не отсутствие ошибки.
- Все evidence_ids существуют в диалоге.
- Все 8 критериев присутствуют ровно по одному разу.

Верни только JSON следующего формата:

{{
  "conversation_id": "{conversation_id}",
  "criteria": [
    {{
      "id": 1,
      "score": "Да",
      "explanation": "Краткое обоснование",
      "evidence_ids": [9]
    }}
  ]
}}

В массиве criteria должны быть ровно 8 объектов,
с id от 1 до 8.

Не добавляй итоговый балл.
Не добавляй текст за пределами JSON.
"""

In [24]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=900,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

input_length = inputs["input_ids"].shape[-1]

response = tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True
)

print(response)

{
  "conversation_id": "p1-ru-001",
  "criteria": [
    {
      "id": 1,
      "score": "Да",
      "explanation": "Оператор поздоровался на русском и представился в первом ответе.",
      "evidence_ids": [9]
    },
    {
      "id": 2,
      "score": "N/A",
      "explanation": "Достаточно информации для ответа уже было в запросе клиента и ответе бота.",
      "evidence_ids": []
    },
    {
      "id": 3,
      "score": "N/A",
      "explanation": "Запрос не требует идентификации личности, так как касается общего доступа к тарифу.",
      "evidence_ids": []
    },
    {
      "id": 4,
      "score": 20,
      "explanation": "Все вопросы клиента решены: указано, где искать тариф, и дана конкретная инструкция.",
      "evidence_ids": [9]
    },
    {
      "id": 5,
      "score": "Да",
      "explanation": "Оператор использовал вежливую формулировку и завершил диалог с уважением.",
      "evidence_ids": [9, 10]
    },
    {
      "id": 6,
      "score": 10,
      "explanation": "Ответ 

In [28]:
import json

result = json.loads(response)

scores = calculate_score(result["criteria"])

result.update(scores)

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "conversation_id": "p1-ru-001",
  "criteria": [
    {
      "id": 1,
      "score": "Да",
      "explanation": "Оператор поздоровался на русском и представился в первом ответе.",
      "evidence_ids": [
        9
      ]
    },
    {
      "id": 2,
      "score": "N/A",
      "explanation": "Достаточно информации для ответа уже было в запросе клиента и ответе бота.",
      "evidence_ids": []
    },
    {
      "id": 3,
      "score": "N/A",
      "explanation": "Запрос не требует идентификации личности, так как касается общего доступа к тарифу.",
      "evidence_ids": []
    },
    {
      "id": 4,
      "score": 20,
      "explanation": "Все вопросы клиента решены: указано, где искать тариф, и дана конкретная инструкция.",
      "evidence_ids": [
        9
      ]
    },
    {
      "id": 5,
      "score": "Да",
      "explanation": "Оператор использовал вежливую формулировку и завершил диалог с уважением.",
      "evidence_ids": [
        9,
        10
      ]
    },
    {
      

In [32]:
from pathlib import Path
from tqdm.auto import tqdm

output_path = Path("results.json")
errors_path = Path("errors.json")

if output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []

if errors_path.exists():
    with open(errors_path, "r", encoding="utf-8") as f:
        errors = json.load(f)
else:
    errors = []

processed_ids = {
    item["conversation_id"]
    for item in results
}

for _, row in tqdm(df.head(5).iterrows(), total=5):

    conversation_id = row["conversation_id"]

    if conversation_id in processed_ids:
        continue

    try:
        result = evaluate_dialogue(row)

        results.append(result)
        processed_ids.add(conversation_id)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    except Exception as e:
        errors.append({
            "conversation_id": conversation_id,
            "error": str(e)
        })

        with open(errors_path, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)

        print(f"Error in {conversation_id}: {e}")

print(f"Successfully evaluated: {len(results)} / {len(df)}")
print(f"Errors recorded: {len(errors)}")

  0%|          | 0/5 [00:00<?, ?it/s]

Successfully evaluated: 5 / 65
Errors recorded: 0
